In [1]:
import os
import random
import numpy as np
from PIL import Image
import imgaug.augmenters as iaa

In [2]:
#Uncomment lower line if you using numpy latest version or it will give error
np.bool = np.bool_

def save_augmented_images(folder_path, images, augmenter, target):
    current_count = len(images)
    image_index = 0  # To ensure unique filenames

    while current_count < target:
        try:
            # Randomly choose an image
            img_path = random.choice(images)
            img = Image.open(img_path).convert("RGB")  # Ensure RGB mode
            img_array = np.array(img)

            # Apply augmentation
            augmented_image = augmenter(image=img_array)
            augmented_image = Image.fromarray(augmented_image)

            # Generate a unique filename
            new_filename = os.path.join(folder_path, f"aug_{image_index}.jpg")
            while os.path.exists(new_filename):  # Ensure uniqueness
                image_index += 1
                new_filename = os.path.join(folder_path, f"aug_{image_index}.jpg")

            # Save the augmented image
            augmented_image.save(new_filename)

            current_count += 1
            image_index += 1  # Update index for next iteration
            
        except Exception as e:
            print(f"Error processing image {img_path}: {e}")

def augment_data(path, size):
    dataset_path = path
    target_count = size  

    augmenter = iaa.Sequential([
        iaa.Fliplr(0.5),  # horizontal flips commonly safe
        iaa.Sometimes(0.25, iaa.Flipud(1.0)),  # occasional vertical flip

        # small rotations / scale / translation
        iaa.Sometimes(0.6, iaa.Affine(
            rotate=(-10, 10),
            scale=(0.95, 1.05),
            translate_percent={"x": (-0.03, 0.03), "y": (-0.03, 0.03)},
            shear=(-3, 3)
        )),

        # small crop/pad
        iaa.Sometimes(0.4, iaa.CropAndPad(percent=(-0.03, 0.06))),

        # slight elastic deformation (very mild)
        iaa.Sometimes(0.25, iaa.ElasticTransformation(alpha=(0.2, 1.0), sigma=0.2)),

        # small blurs/sharpen/contrast
        iaa.SomeOf((0, 1), [
            iaa.GaussianBlur((0.0, 0.8)),
            iaa.Sharpen(alpha=(0.0, 0.25), lightness=(0.9, 1.1)),
            iaa.LinearContrast((0.9, 1.1))
        ], random_order=True),

        # gentle brightness/hue variation (avoid massive color shifts)
        iaa.Sometimes(0.5, iaa.Multiply((0.9, 1.1))),
        iaa.Sometimes(0.2, iaa.AddToHueAndSaturation((-8, 8))),

        # light noise / occlusion
        iaa.Sometimes(0.15, iaa.AdditiveGaussianNoise(scale=(0, 0.01*255))),
        iaa.Sometimes(0.1, iaa.CoarseDropout((0.001, 0.01), size_percent=(0.01, 0.05)))
    ], random_order=True)

    # Detect folders in the dataset directory
    folders = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))]

    # Augment images for each detected folder
    for folder in folders:
        folder_path = os.path.join(dataset_path, folder)
        images = [os.path.join(folder_path, img) for img in os.listdir(folder_path) if img.endswith(('.png', '.jpg', '.jpeg'))]

        # Perform augmentation if needed
        if len(images) < target_count:
            print(f"Augmenting {folder} from {len(images)} to {target_count} images.")
            save_augmented_images(folder_path, images, augmenter, target_count)
        else:
            print(f"{folder} already has {len(images)} images or more.")

    print("Data augmentation completed!")

In [3]:
augment_data(r"Segmented Dataset\HAM10000_organized_masked_out", 6705)

Augmenting akiec from 327 to 6705 images.
Augmenting bcc from 514 to 6705 images.
Augmenting bkl from 1099 to 6705 images.
Augmenting df from 115 to 6705 images.
Augmenting mel from 1113 to 6705 images.
nv already has 6705 images or more.
Augmenting vasc from 142 to 6705 images.
Data augmentation completed!


In [4]:
augment_data(r"Segmented Dataset\MILK10k_Organized_masked_out", 1600)

Augmenting AKIEC from 606 to 1600 images.
BCC already has 5044 images or more.
Augmenting BEN_OTH from 88 to 1600 images.
Augmenting BKL from 1088 to 1600 images.
Augmenting DF from 104 to 1600 images.
Augmenting INF from 100 to 1600 images.
Augmenting MAL_OTH from 18 to 1600 images.
Augmenting MEL from 900 to 1600 images.
Augmenting NV from 1492 to 1600 images.
Augmenting SCCKA from 946 to 1600 images.
Augmenting VASC from 94 to 1600 images.
Data augmentation completed!
